In [ ]:
import pandas
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import cv2
import os
import re
from PIL import Image

In [ ]:
NUM_CLASSES = 18
BATCH_SIZE = 32
THRESHOLD = 0.7
EPOCH = 5
LR = 1e-4
DECAY = 1.0e-4
IMG_SIZE = (224, 224)
SEED = 2023
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TEXT_WEIGHT = 1
IMG_WEIGHT = 1

In [ ]:
torch.manual_seed(SEED)
DEVICE

In [ ]:
%pip install  gdown
%pip install sentence_transformers

In [ ]:
!gdown 1hUqu1mbFeTEfBvl-7fc56fHFfCSzIktD
!unzip -qq ml1m.zip -d ml1m

In [ ]:
users = pandas.read_csv('ml1m/content/dataset/users.dat', sep='::',
                        engine='python',
                        names=['userid', 'gender', 'age', 'occupation', 'zip']).set_index('userid')
ratings = pandas.read_csv('ml1m/content/dataset/ratings.dat', engine='python',
                          sep='::', names=['userid', 'movieid', 'rating', 'timestamp'])
movies_train = pandas.read_csv('ml1m/content/dataset/movies_train.dat', engine='python',
                         sep='::', names=['movieid', 'title', 'genre'], encoding='latin-1', index_col=False).set_index('movieid')
movies_test = pandas.read_csv('ml1m/content/dataset/movies_test.dat', engine='python',
                         sep='::', names=['movieid', 'title', 'genre'], encoding='latin-1', index_col=False).set_index('movieid')
movies_train['genre'] = movies_train.genre.str.split('|')
movies_test['genre'] = movies_test.genre.str.split('|')

users.age = users.age.astype('category')
users.gender = users.gender.astype('category')
users.occupation = users.occupation.astype('category')
ratings.movieid = ratings.movieid.astype('category')
ratings.userid = ratings.userid.astype('category')

In [ ]:
folder_img_path = 'ml1m/content/dataset/ml1m-images'
movies_train['id'] = movies_train.index
movies_train.reset_index(inplace=True)
movies_train['img_path'] = movies_train.apply(lambda row: os.path.join(folder_img_path, f'{row.id}.jpg'), axis = 1)
movies_train

In [ ]:
folder_img_path = 'ml1m/content/dataset/ml1m-images'
movies_test['id'] = movies_test.index
movies_test.reset_index(inplace=True)
movies_test['img_path'] = movies_test.apply(lambda row: os.path.join(folder_img_path, f'{row.id}.jpg'), axis = 1)

In [ ]:
def process_movie_title(title):
    if ',' in title:
        comma_index = title.find(',')
    
        part1 = title[:comma_index].strip() if comma_index != -1 else title.strip()
        part2 = title[comma_index + 1:-6].strip() if comma_index != -1 else ''
        part3 = title[-6:]
        return part2 +" "+ part1 +" "+ part3
    else:
        return title
sample_title = "Manchurian Candidate, The (1962)"
processed_title = process_movie_title(sample_title)
print(processed_title)

In [ ]:
movies_train['title'] = movies_train['title'].apply(process_movie_title)
movies_test['title'] = movies_test['title'].apply(process_movie_title)
movies_train.head(), movies_test.head()

In [ ]:
from nltk import wordpunct_tokenize

def tokenize(text):
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    tokens = wordpunct_tokenize(text)
    tokens = tokens[:-1] # remove last token because it is the year which maybe is not useful
    return tokens

def create_vocab():
    df = movies_train.copy()
    arr_title = df['title'].tolist()
    vocab = set()
    for title in arr_title:
        tokens = tokenize(title)
        vocab.update(tokens)
    vocab = list(vocab)
    pad_token = '<PAD>'
    unk_token = '<UNK>'
    vocab.append(pad_token)
    vocab.append(unk_token)
    return vocab

In [ ]:
from typing import *
import numpy as np

from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
def extract_text_feat(
    title_list,
    model,
    batch_size = 8,
    device=DEVICE,
    show_progress_bar=True):
    
    p_vectors = model.encode(title_list, show_progress_bar=show_progress_bar,
                            batch_size=batch_size,convert_to_numpy=True,
                            normalize_embeddings=True)
    print('output shape', p_vectors.shape)
    return p_vectors

In [ ]:
train_title = movies_train['title']
test_title = movies_test['title']
print(len(train_title), len(test_title))

train_title_list = []
test_title_list = []

for i in train_title:
    train_title_list.append(i[:-7])  
    
for i in test_title:
    test_title_list.append(i[:-7]) # -7 due to rm year
    
train_text_feats = extract_text_feat(train_title_list, sbert)
test_text_feats = extract_text_feat(test_title_list, sbert)

In [ ]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
)

class MLDataset(Dataset):
    def __init__(self, is_train=True):
        if is_train:
            self.data =  movies_train
            self.text_feat = train_text_feats
        else:
            self.data = movies_test
            self.text_feat = test_text_feats
            
        self.data['title_tokens'] = [tokenize(x) for x in self.data.title]

        # create vocab
        vocab = create_vocab()
        pad_token = '<PAD>'
        unk_token = '<UNK>'
        self.token2idx = {token: idx for idx, token in enumerate(vocab)}

        # Create a binary vector for each word in each sentence
        MAX_LENGTH = 7
        vectors = []
        for title_tokens in self.data.title_tokens.tolist():
            if len(title_tokens) < MAX_LENGTH:
                num_pad = MAX_LENGTH - len(title_tokens)
                for idx in range(num_pad):
                    title_tokens.append(pad_token)
            else:
                title_tokens = title_tokens[:MAX_LENGTH]
            title_vectors = []
            for word in title_tokens:
                binary_vector = np.zeros(len(vocab))
                if word in vocab:
                    binary_vector[self.token2idx[word]] = 1
                else:
                    binary_vector[self.token2idx[unk_token]] = 1
                title_vectors.append(binary_vector)

            vectors.append(np.array(title_vectors))
        self.data['vectors'] = vectors

        # label genre
        with open('ml1m/content/dataset/genres.txt', 'r') as f:
            genre_all = f.readlines()
            genre_all = [x.replace('\n','') for x in genre_all]
        self.genre2idx = {genre:idx for idx, genre in enumerate(genre_all)}
        
    def __getitem__(self, index):
        img_path = self.data.iloc[index].img_path
        genre = self.data.iloc[index].genre


        title_tensor = self.text_feat[index]
        # preprocess img
        if os.path.exists(img_path):
            img = Image.open(img_path).convert('RGB') # makesure 3 channels r g b
            img = transform(img)
        else:
            img = torch.randn(3, 224, 224)
        img_tensor = img
        # preprocess label
        genre_vector = np.zeros(len(self.genre2idx))

        for g in genre:
            genre_vector[self.genre2idx[g]] = 1
        genre_tensor = torch.from_numpy(genre_vector).float()

        return title_tensor, img_tensor, genre_tensor

    def __len__(self):
        return len(self.data)

In [ ]:
train_set = MLDataset(is_train=True)
test_set = MLDataset(is_train=False)

train_dataloader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
for text, img, label in train_dataloader:
    print(text.shape, img.shape, label.shape)
    break

In [ ]:
import torchvision.models as models

class MVLModel(nn.Module):
    def __init__(self):
        super(MVLModel, self).__init__()
        # img layers
        self.vgg16_model = models.vgg16(pretrained=True, progress=True)
        self.act = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.5)
        self.lin2 = nn.Linear(1000, 64, bias=True)
        
        # text layers
        self.act2 = nn.ReLU(inplace=True)
        self.dropout2 = nn.Dropout(0.5)
        self.lin3 = nn.Linear(384, 64, bias=True)
        
        
        # fuse layers
        self.lin4 = nn.Linear(64 * 2, NUM_CLASSES, bias=True)
        
        
    def forward(self, x, y):
        '''
        x: img
        y: text
        '''
        text_out = self.encode_text(y)
        img_out = self.encode_img(x)
        
        # attention concat
        feats = torch.cat([img_out * IMG_WEIGHT, text_out * TEXT_WEIGHT], dim=1)
        feats = self.dropout2(feats)
        out = self.lin4(feats)
        return out
    
    def encode_text(self, text):
        out = self.lin3(text)
        out = self.act2(out)
        return out
        
    def encode_img(self, img):
        out = self.vgg16_model(img)
        out = self.act(out)
        out = self.dropout(out)
        out = self.lin2(out)
        return out
        
    
    @torch.no_grad()
    def evaluate(self, x):
        with torch.no_grad():
            out = self(x) > THRESHOLD
        return out

In [ ]:
mvlModel = MVLModel().to(DEVICE)
print(mvlModel)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mvlModel.parameters(), lr=LR, weight_decay=DECAY)

In [ ]:
from tqdm import tqdm

def train(model, dataloader):
    model.train(True)
    for epoch in range(EPOCH):
        pbar = tqdm(enumerate(dataloader), total=len(dataloader))
        
        for i, batch_data in pbar:
            text, img, label = batch_data
            text = text.to(DEVICE)
            img = img.to(DEVICE)
            label = label.to(DEVICE)
            
            optimizer.zero_grad()
            out = model(img, text)
            loss = loss_fn(out, label)
            loss.backward()
            optimizer.step()
            pbar.set_description("loss %.4f" %(loss.detach()))  
        

In [ ]:
train(mvlModel, train_dataloader)

In [ ]:
from torchmetrics.classification import MultilabelF1Score
from torchmetrics.classification import MultilabelRecall
from torchmetrics.classification import MultilabelPrecision
from torchmetrics.classification import MultilabelAccuracy

def P_at_K(k, pred, truth):
    # print(pred)
    _, indices = torch.topk(pred, k=k)
    correct = 0
    for id in indices:
        if truth[id] > 0:
            correct += 1
    return correct / k

def AP_at_K(k, pred, truth):
    AP = 0
    for i in range(1, k+1):
        AP += P_at_K(i, pred, truth) 
    return AP / k

def MAP_at_K(k, pred_list, truth_list):
    MAP = 0
    for i in range(len(pred_list)):
        MAP += AP_at_K(k, pred_list[i], truth_list[i])
    return MAP / len(pred_list)


def evaluate(model, dataloader):
    model.train(False)
    with torch.no_grad(): 
        preds = []
        gdtruth = []
        pbar = tqdm(enumerate(dataloader), total=len(dataloader))
        for i, batch_data in pbar:
            text, img, label = batch_data
            text = text.to(DEVICE)
            img = img.to(DEVICE)
            label = label.to(DEVICE)
            # out = model(img) > THRESHOLD
            out = model(img, text)
            
            preds.append(out.to('cpu'))
            gdtruth.append(label.to('cpu'))
            
        preds = torch.cat(preds, dim=0)
        gdtruth = torch.cat(gdtruth, dim=0)
    
    metric_f1 = MultilabelF1Score(num_labels=NUM_CLASSES, threshold=THRESHOLD, average='micro')
    metric_re = MultilabelRecall(num_labels=NUM_CLASSES, threshold=THRESHOLD, average='micro')
    metric_pre = MultilabelPrecision(num_labels=NUM_CLASSES, threshold=THRESHOLD, average='micro')
    metric_acc = MultilabelAccuracy(num_labels=NUM_CLASSES, threshold=THRESHOLD, average='micro')
    
    map1 = MAP_at_K(1, preds, gdtruth)
    map2 = MAP_at_K(2, preds, gdtruth)
    map3 = MAP_at_K(3, preds, gdtruth)
    map4 = MAP_at_K(4, preds, gdtruth)
    
    f1_s = metric_f1(preds, gdtruth)
    re_s = metric_re(preds, gdtruth)
    pre_s = metric_pre(preds, gdtruth)
    acc_s = metric_acc(preds, gdtruth)
    
    
    return (
        f1_s, re_s, acc_s, pre_s, map1, map2, map3, map4
    )


In [ ]:
f1, re, acc, pre, map1, map2, map3, map4 = evaluate(mvlModel, test_dataloader)
print("f1: %.4f, precision: %.4f, recall: %.4f, accuracy: %.4f" %(f1, pre, re, acc))
print("map@1: %.4f" %(map1))
print("map@2: %.4f" %(map2))
print("map@3: %.4f" %(map3))
print("map@4: %.4f" %(map4))